# cGAN + Structure Conditioning — 128×128  (Model C)
### Model C: condition the generator on a per-image **structure map** instead of scoring against an unaligned target

Models A and B condition the generator on the **class label only** and add a
*pixel* (A) or *pixel + MediaPipe-landmark* (B) loss against an **unaligned** real
image — which drags outputs toward the per-class mean (regress-to-mean). **Model C
changes the conditioning, not just the loss:**

- For every image we compute a 3-channel **structure map** — **Canny edges +
  silhouette + distance transform** — with **100% coverage** (no detector to fail,
  unlike MediaPipe in Model B).
- The generator takes `(structure, label, noise)` → image (an encoder–decoder).
- The discriminator is **paired**: it judges `(image, structure, label)` triples.
- Because the structure map **and** the real target come from the **same** image,
  spatial correspondence is restored → no regress-to-mean → diversity recovers.

**Shared with A/B:** SAGAN-style backbone ideas, SpectralNorm on D, asymmetric LR,
label smoothing, mixed precision. **Extra dependency:** only OpenCV (Canny /
distance transform) — no MediaPipe.

> In the local A/B/C run this wins decisively (GAN-test ≈ 0.95 vs ~0.5 for A/B) and
> a **held-out structure test** (Section 10) confirms it generalizes rather than
> memorizes. See `reports/` and `experiments/`.

## Section 1 — Environment

In [ ]:
# ── Install (pinned for reproducibility) ──────────────────────────────────────
# Model C needs NO MediaPipe — structure maps come from OpenCV.
!pip install tensorflow==2.19.0 opencv-python --quiet
!pip install pytorch-fid scikit-image scikit-learn tqdm --quiet
!pip install torch torchvision --quiet
print("All packages installed.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import os, json, csv, warnings
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
from collections import Counter
from PIL import Image

import tensorflow as tf
from tensorflow.keras import layers
from sklearn.preprocessing import LabelEncoder
from skimage.metrics import structural_similarity as ssim_fn

import cv2

warnings.filterwarnings('ignore', category=UserWarning)
print(f"TensorFlow : {tf.__version__}")
print(f"OpenCV     : {cv2.__version__}")
print(f"GPU        : {tf.config.list_physical_devices('GPU')}")

In [ ]:
# ── Global reproducibility ─────────────────────────────────────────────────────
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
print(f"Global seed: {RANDOM_SEED}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
        print(f"GPU memory growth enabled: {gpus[0].name}")
    except RuntimeError as e:
        print(f"GPU config warning: {e}")

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
DRIVE_BASE     = "/content/drive/MyDrive/cgan_C_128struct"
DATA_PATH      = "/content/drive/MyDrive/ArASL_Database_54K_Final/ArASL_Database_54K_Final"

CHECKPOINT_DIR = os.path.join(DRIVE_BASE, "checkpoints")
SAMPLES_DIR    = os.path.join(DRIVE_BASE, "samples")
EVAL_DIR       = os.path.join(DRIVE_BASE, "eval")
REAL_FID_DIR   = os.path.join(EVAL_DIR,   "fid_real")
HISTORY_DIR    = os.path.join(DRIVE_BASE, "history")
PLOTS_DIR      = os.path.join(DRIVE_BASE, "plots")
PROGRESS_FILE  = os.path.join(CHECKPOINT_DIR, "progress.json")

for d in [CHECKPOINT_DIR, SAMPLES_DIR, EVAL_DIR, REAL_FID_DIR, HISTORY_DIR, PLOTS_DIR]:
    os.makedirs(d, exist_ok=True)

print("All directories ready.")
for d in [CHECKPOINT_DIR, SAMPLES_DIR, EVAL_DIR, HISTORY_DIR, PLOTS_DIR]:
    print(f"  {d}")

## Section 2 — Hyperparameters

In [ ]:
# ════════════════════════════════════════════════════════
#  CONFIGURATION — change only this cell
# ════════════════════════════════════════════════════════

# Architecture
Z_DIM        = 128    # latent noise dimension (style only — structure carries shape)
IMG_SIZE     = 128
IMG_CHANNELS = 1      # grayscale
COND_CH      = 3      # structure map channels: edge + silhouette + distance

# Training
EPOCHS         = 50
BATCH_SIZE     = 32
LR_G           = 2e-4
LR_D           = 1e-4  # asymmetric: D converges faster
LABEL_SMOOTH   = 0.9   # real-label smoothing

# Mixed precision (flip OFF to reproduce the exact float32 path)
USE_MIXED_PRECISION = True
USE_XLA             = False

# Held-out split (for the structure generalization test, Section 10)
EVAL_FRACTION    = 0.10
MIN_EVAL_PER_CLS = 30

# Evaluation
N_FID_PER_CLASS  = 60
N_FID_SEEDS      = 5
N_DIV_PER_CLASS  = 10
SAVE_EVERY_N     = 5

# Structure-map (Canny) thresholds
CANNY_LO, CANNY_HI = 60, 160

if USE_MIXED_PRECISION and gpus:
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
    print("Mixed precision: mixed_float16 ON")
else:
    print("Mixed precision: OFF (float32)")

print("Model C — cGAN 128×128 (Structure-conditioned)")
print(f"  IMG_SIZE   = {IMG_SIZE}  COND_CH = {COND_CH} (edge+silhouette+distance)")
print(f"  BATCH_SIZE = {BATCH_SIZE}")
print(f"  LR_G={LR_G}, LR_D={LR_D}  (asymmetric)")
print(f"  Conditioning = per-image structure map (100% coverage, no MediaPipe)")

## Section 3 — Data Loading

In [ ]:
# ── Load images at 128×128 ─────────────────────────────────────────────────────
VALID_EXT = ('.png', '.jpg', '.jpeg')
images_list, labels_list = [], []
load_errors = 0

print(f"Loading from: {DATA_PATH}  (resizing to {IMG_SIZE}×{IMG_SIZE})")
for subfolder in tqdm(sorted(os.listdir(DATA_PATH)), desc="Classes"):
    spath = os.path.join(DATA_PATH, subfolder)
    if not os.path.isdir(spath):
        continue
    for fname in sorted(os.listdir(spath)):
        if not fname.lower().endswith(VALID_EXT):
            continue
        try:
            raw = tf.io.read_file(os.path.join(spath, fname))
            img = tf.image.decode_png(raw, channels=1)
            img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])   # bilinear → 128×128
            img = tf.cast(img, tf.float32)
            img = (img - 127.5) / 127.5                        # [-1, 1]
            images_list.append(img.numpy())
            labels_list.append(subfolder)
        except Exception:
            load_errors += 1

print(f"Loaded : {len(images_list)} images")
print(f"Errors : {load_errors}")

In [ ]:
# ── NumPy arrays + label encoding ─────────────────────────────────────────────
images_array = np.array(images_list, dtype=np.float32)
labels_array = np.array(labels_list)

label_encoder = LabelEncoder()
labels_int    = label_encoder.fit_transform(labels_array).astype(np.int64)
num_classes   = len(label_encoder.classes_)
label_to_idx  = {l: i for i, l in enumerate(label_encoder.classes_)}
idx_to_label  = {i: l for l, i in label_to_idx.items()}

np.save(os.path.join(CHECKPOINT_DIR, "classes.npy"), label_encoder.classes_)

print(f"Images shape : {images_array.shape}")
print(f"Pixel range  : [{images_array.min():.3f}, {images_array.max():.3f}]")
print(f"Classes      : {num_classes}")
print(f"Class list   : {list(label_encoder.classes_)}")

In [ ]:
# ── Shuffle + held-out split (eval structures are NEVER trained on) ────────────
rng  = np.random.default_rng(RANDOM_SEED)
perm = rng.permutation(len(images_array))
images_array, labels_int = images_array[perm], labels_int[perm]

n_eval = max(num_classes * MIN_EVAL_PER_CLS, int(EVAL_FRACTION * len(images_array)))
X_tr, y_tr = images_array[:-n_eval], labels_int[:-n_eval]
X_ev, y_ev = images_array[-n_eval:], labels_int[-n_eval:]
print(f"train : {X_tr.shape}  ({len(X_tr)} images)")
print(f"eval  : {X_ev.shape}  ({len(X_ev)} held-out images for the structure test)")

## Section 4 — Structure Maps (the Model C conditioning signal)

For each image we build a 3-channel map — **Canny edges**, **silhouette** (Otsu),
**distance transform** of the silhouette. This is what the generator is conditioned
on. Unlike MediaPipe (Model B, ~2% detection at low res) it has **100% coverage**.

In [ ]:
# ── Structure map: edge + silhouette + distance-transform ─────────────────────
def structure_map(img_norm):
    """img_norm: (H,W,1) in [-1,1]  ->  (H,W,3) structure map in [-1,1]."""
    g = ((img_norm[:, :, 0] + 1) * 127.5).clip(0, 255).astype(np.uint8)
    edge = cv2.Canny(g, CANNY_LO, CANNY_HI)
    _, sil = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if sil.mean() > 127:                       # keep hand as foreground
        sil = 255 - sil
    dist = cv2.normalize(cv2.distanceTransform(sil, cv2.DIST_L2, 3),
                         None, 0, 255, cv2.NORM_MINMAX)
    return (np.stack([edge, sil, dist], -1).astype(np.float32) / 127.5) - 1.0

C_tr = np.stack([structure_map(x) for x in tqdm(X_tr, desc="structure maps")]).astype(np.float32)
coverage = float(np.mean([np.any(c > -0.99) for c in C_tr]))
print("condition maps:", C_tr.shape, "| coverage:", coverage)

In [ ]:
# ── Quick visual sanity check: image -> [edge | silhouette | distance] ────────
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for col in range(4):
    img = (X_tr[col, :, :, 0] * 127.5 + 127.5).clip(0, 255).astype(np.uint8)
    sm  = (C_tr[col] * 127.5 + 127.5).clip(0, 255).astype(np.uint8)
    axes[0, col].imshow(img, cmap='gray'); axes[0, col].set_title(idx_to_label[y_tr[col]]); axes[0, col].axis('off')
    axes[1, col].imshow(sm); axes[1, col].set_title("edge|sil|dist"); axes[1, col].axis('off')
axes[0, 0].set_ylabel("real", fontsize=12)
plt.suptitle("Model C conditioning: real image → 3-channel structure map", y=1.02)
plt.tight_layout(); plt.show()

## Section 5 — Model Architecture (structure-conditioned)

**Generator** is an encoder–decoder: it *encodes the structure map*, fuses it with
the label embedding and noise, then decodes to a 128×128 image. **Discriminator**
is *paired* — it sees `(image, structure, label)` and asks "does this image match
*this* structure and label?" (pix2pix-style conditional adversarial supervision).

In [ ]:
# ── Generator block (transposed conv, ×2 upsample) ────────────────────────────
def gen_block(x, f):
    x = layers.Conv2DTranspose(f, 4, 2, padding="same", use_bias=False)(x)
    return layers.ReLU()(layers.BatchNormalization()(x))

# ── Structure-conditioned generator: (condition, label, noise) -> image ───────
def build_generator(num_classes):
    c = tf.keras.Input((IMG_SIZE, IMG_SIZE, COND_CH))   # structure map
    l = tf.keras.Input((num_classes,))                  # one-hot label
    n = tf.keras.Input((Z_DIM,))                        # noise (style)
    # encode the structure map  128 -> 8
    e = c
    for f in [64, 128, 256, 512]:
        e = layers.LeakyReLU(0.2)(layers.BatchNormalization()(
            layers.Conv2D(f, 4, 2, padding="same", use_bias=False)(e)))
    e = layers.Flatten()(e)                             # 8*8*512
    le = layers.LeakyReLU(0.2)(layers.Dense(128, use_bias=False)(l))
    x = layers.Dense(8 * 8 * 512, use_bias=False)(layers.Concatenate()([e, n, le]))
    x = layers.ReLU()(layers.BatchNormalization()(x))
    x = layers.Reshape((8, 8, 512))(x)
    x = gen_block(x, 256); x = gen_block(x, 128); x = gen_block(x, 64); x = gen_block(x, 32)  # 8->128
    o = layers.Activation("tanh", dtype="float32")(
        layers.Conv2D(IMG_CHANNELS, 3, padding="same", dtype="float32")(x))
    return tf.keras.Model([c, l, n], o, name="generator_C")

# ── Paired discriminator: (image, condition, label) -> logit ──────────────────
def build_discriminator(num_classes):
    SN  = layers.SpectralNormalization
    img = tf.keras.Input((IMG_SIZE, IMG_SIZE, IMG_CHANNELS))
    c   = tf.keras.Input((IMG_SIZE, IMG_SIZE, COND_CH))
    l   = tf.keras.Input((num_classes,))
    lp  = layers.Reshape((IMG_SIZE, IMG_SIZE, 1))(layers.Dense(IMG_SIZE * IMG_SIZE)(l))
    x   = layers.Concatenate()([img, c, lp])
    for f in [64, 128, 256, 512, 512]:
        x = layers.LeakyReLU(0.2)(SN(layers.Conv2D(f, 4, 2, padding="same"))(x))
    o = layers.Dense(1, dtype="float32")(layers.Flatten()(x))
    return tf.keras.Model([img, c, l], o, name="discriminator_C")

In [ ]:
# ── Build + inspect ───────────────────────────────────────────────────────────
generator     = build_generator(num_classes)
discriminator = build_discriminator(num_classes)
print(f"Generator     params: {generator.count_params():,}")
print(f"Discriminator params: {discriminator.count_params():,}")

## Section 6 — Loss & Optimizer Helpers (mixed-precision safe)

Model C has **no separate pixel/landmark loss term**: the supervision is the
adversarial loss from the *paired* discriminator plus an L1 to the **aligned**
target (same image the structure came from). That alignment is what removes the
regress-to-mean problem A and B suffer from.

In [ ]:
bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)
LAMBDA_L1 = 5.0   # weight on the ALIGNED L1 (target == the image the structure came from)

def make_opt(lr):
    o = tf.keras.optimizers.Adam(lr, beta_1=0.5, clipnorm=1.0)
    return tf.keras.mixed_precision.LossScaleOptimizer(o) if USE_MIXED_PRECISION else o

def apply_loss(opt, loss, tape, vs):
    if USE_MIXED_PRECISION:
        g = tape.gradient(opt.scale_loss(loss), vs)
        opt.apply(g, vs)   # Keras 3 unscales internally
    else:
        g = tape.gradient(loss, vs)
        opt.apply_gradients(zip(g, vs))

g_opt = make_opt(LR_G)
d_opt = make_opt(LR_D)
def onehot(y): return tf.one_hot(y, num_classes)
print("Optimizers + losses ready (LossScaleOptimizer:", USE_MIXED_PRECISION, ")")

## Section 7 — Visualisation Utilities

In [ ]:
# ── Fixed eval structures (from held-out set) for visual tracking across epochs ─
TEST_LABELS = [l for l in ["bb", "ain", "gaaf", "al", "ha"] if l in label_to_idx]
if not TEST_LABELS:
    TEST_LABELS = [idx_to_label[i] for i in range(min(5, num_classes))]

# pick one held-out image per test label -> its structure map is the condition
_vis_idx = []
for lbl in TEST_LABELS:
    pool = np.where(y_ev == label_to_idx[lbl])[0]
    _vis_idx.append(int(pool[0]) if len(pool) else int(np.where(y_tr == label_to_idx[lbl])[0][0]))
VIS_COND  = np.stack([structure_map(X_ev[i]) for i in _vis_idx]).astype(np.float32)
tf.random.set_seed(RANDOM_SEED)
VIS_NOISE = tf.random.normal([len(TEST_LABELS), Z_DIM], seed=RANDOM_SEED)
print(f"Test labels : {TEST_LABELS}")

In [ ]:
def generate_and_save_images(model, epoch, save_dir=None):
    if save_dir is None: save_dir = SAMPLES_DIR
    n = len(TEST_LABELS)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1: axes = [axes]
    for i, lbl in enumerate(TEST_LABELS):
        loh = tf.one_hot([label_to_idx[lbl]], depth=num_classes)
        gen = model([VIS_COND[i:i+1], loh, VIS_NOISE[i:i+1]], training=False)
        img = (gen[0, :, :, 0].numpy() * 127.5 + 127.5).clip(0, 255).astype("uint8")
        axes[i].imshow(img, cmap='gray', vmin=0, vmax=255)
        axes[i].set_title(lbl, fontsize=13, fontweight='bold'); axes[i].axis('off')
    plt.suptitle(f"cGAN-C 128px (Structure-cond) — Epoch {epoch}", fontsize=14, y=1.02)
    plt.tight_layout()
    p = os.path.join(save_dir, f"epoch_{epoch:04d}.png")
    plt.savefig(p, dpi=300, bbox_inches='tight'); plt.show(); plt.close()
    return p

## Section 8 — Training Loop

In [ ]:
# ── tf.data pipeline (image, label, structure map) ────────────────────────────
def make_ds(arrays):
    ds = tf.data.Dataset.from_tensor_slices(arrays)
    ds = ds.shuffle(len(arrays[0]), seed=RANDOM_SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds((X_tr, y_tr, C_tr))

@tf.function(jit_compile=USE_XLA)
def train_step(real, y, cond):
    oh = onehot(y)
    nz = tf.random.normal([tf.shape(real)[0], Z_DIM])
    # ── Discriminator: paired (image, structure, label) ──
    with tf.GradientTape() as t:
        fake = generator([cond, oh, nz], training=True)
        d_real = discriminator([real, cond, oh], training=True)
        d_fake = discriminator([fake, cond, oh], training=True)
        d_loss = (bce(tf.ones_like(d_real) * LABEL_SMOOTH, d_real)
                  + bce(tf.zeros_like(d_fake), d_fake))
    apply_loss(d_opt, d_loss, t, discriminator.trainable_variables)
    # ── Generator: fool D + ALIGNED L1 (target == the image the structure came from) ──
    nz = tf.random.normal([tf.shape(real)[0], Z_DIM])
    with tf.GradientTape() as t:
        fake = generator([cond, oh, nz], training=True)
        f    = discriminator([fake, cond, oh], training=True)
        g_adv = bce(tf.ones_like(f), f)
        g_l1  = LAMBDA_L1 * tf.reduce_mean(tf.abs(fake - real))
        g_loss = g_adv + g_l1
    apply_loss(g_opt, g_loss, t, generator.trainable_variables)
    return d_loss, g_loss, g_adv, g_l1

In [ ]:
# ── Run training ──────────────────────────────────────────────────────────────
hist = {"d": [], "g": [], "g_adv": [], "g_l1": []}
print(f"Starting Model C — structure-conditioned cGAN 128×128 · {EPOCHS} epochs")
for ep in range(1, EPOCHS + 1):
    dl = gl = ga = l1 = k = 0.0
    for real, y, cond in train_ds:
        d, g, a, p = train_step(real, y, cond)
        dl += float(d); gl += float(g); ga += float(a); l1 += float(p); k += 1
    hist["d"].append(dl / k); hist["g"].append(gl / k)
    hist["g_adv"].append(ga / k); hist["g_l1"].append(l1 / k)
    print(f"  ep{ep:02d}/{EPOCHS}  D={dl/k:.3f}  G={gl/k:.3f}  (adv={ga/k:.3f}  L1={l1/k:.3f})")
    if ep % SAVE_EVERY_N == 0 or ep == 1 or ep == EPOCHS:
        generate_and_save_images(generator, ep)
        generator.save_weights(os.path.join(CHECKPOINT_DIR, "generator_C.weights.h5"))
        discriminator.save_weights(os.path.join(CHECKPOINT_DIR, "discriminator_C.weights.h5"))
print("Training complete.")

## Section 9 — Loss Curves

In [ ]:
def plot_losses(hist, save_dir=PLOTS_DIR):
    ep = range(1, len(hist["d"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    ax[0].plot(ep, hist["d"], label="D loss"); ax[0].plot(ep, hist["g"], label="G loss")
    ax[0].set_title("Adversarial losses"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].plot(ep, hist["g_adv"], label="G adversarial"); ax[1].plot(ep, hist["g_l1"], label="G aligned-L1")
    ax[1].set_title("Generator terms"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "loss_curves.png"), dpi=150, bbox_inches="tight"); plt.show()

plot_losses(hist)

## Section 10 — Evaluation Suite

For Model C we report:
- **GAN-test (recognition):** a classifier trained on *real* images, tested on
  *generated* ones — how recognizable the fakes are.
- **Diversity:** mean intra-class L1 spread of generations.
- **SSIM / FID:** fidelity to real data.
- **⭐ Held-out structure test:** feed C structure maps from images it **never
  trained on**. If recognition stays high (small *generalization gap*) and SSIM to
  the true target is high, C learned the *structure → image* mapping rather than
  memorizing training images.

In [ ]:
# ── Reference classifier trained on REAL images (for GAN-test recognition) ────
clf = tf.keras.Sequential([
    tf.keras.Input((IMG_SIZE, IMG_SIZE, 1)),
    layers.Conv2D(32, 3, 2, "same"), layers.LeakyReLU(0.2),
    layers.Conv2D(64, 3, 2, "same"), layers.LeakyReLU(0.2),
    layers.Conv2D(128, 3, 2, "same"), layers.LeakyReLU(0.2),
    layers.GlobalAveragePooling2D(),
    layers.Dense(num_classes, dtype="float32")])
clf.compile(optimizer="adam",
            loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
            metrics=["accuracy"])
clf.fit(X_tr, y_tr, epochs=8, batch_size=64, validation_data=(X_ev, y_ev), verbose=0)
clf_ref = float(clf.evaluate(X_ev, y_ev, verbose=0)[1])
print(f"Reference classifier accuracy on real held-out: {clf_ref:.4f}  (chance = {1/num_classes:.3f})")

In [ ]:
# ── Generate from TRAINING structures (balanced per class) ────────────────────
def gen_from_train_structures(G, n_per=40, seed=0):
    tf.random.set_seed(seed); out, ys = [], []
    for c in range(num_classes):
        pool = np.where(y_tr == c)[0]
        if not len(pool): continue
        idx = np.random.default_rng(seed + c).choice(pool, n_per)
        oh  = onehot(np.full(n_per, c))
        nz  = tf.random.normal([n_per, Z_DIM], seed=seed * 100 + c)
        out.append(G([C_tr[idx], oh, nz], training=False).numpy()); ys.append(np.full(n_per, c))
    return np.concatenate(out), np.concatenate(ys)

def recognition(G, n_per=40):
    f, ys = gen_from_train_structures(G, n_per=n_per)
    pred = clf.predict(f, verbose=0).argmax(1)
    return float((pred == ys).mean())

def diversity(G, n_per=N_DIV_PER_CLASS, seed=1):
    tf.random.set_seed(seed); ds = []
    for c in range(num_classes):
        pool = np.where(y_tr == c)[0]
        if not len(pool): continue
        idx = np.random.default_rng(seed + c).choice(pool, n_per)
        oh  = onehot(np.full(n_per, c)); nz = tf.random.normal([n_per, Z_DIM], seed=seed + c)
        f   = G([C_tr[idx], oh, nz], training=False).numpy().reshape(n_per, -1)
        ds += [float(np.mean(np.abs(f[i] - f[j]))) for i in range(n_per) for j in range(i + 1, n_per)]
    return float(np.mean(ds))

rec_m = recognition(generator)
div_m = diversity(generator)
print(f"GAN-test recognition : {rec_m:.4f}   (vs real classifier {clf_ref:.4f})")
print(f"Diversity            : {div_m:.4f}   (mean intra-class L1; higher = more diverse)")

In [ ]:
# ── ⭐ Held-out structure test: does C generalize or just copy? ───────────────
def heldout_structure_test(G, seed=0):
    # structures from EVAL images the generator never saw
    C_ev = np.stack([structure_map(x) for x in X_ev]).astype(np.float32)
    oh   = onehot(y_ev); nz = tf.random.normal([len(X_ev), Z_DIM], seed=seed)
    fake_ev = G([C_ev, oh, nz], training=False).numpy()
    rec_heldout = float((clf.predict(fake_ev, verbose=0).argmax(1) == y_ev).mean())

    # recognition from the SAME number of TRAINING structures (for the gap)
    idx  = np.random.default_rng(seed).choice(len(X_tr), len(X_ev))
    oh2  = onehot(y_tr[idx]); nz2 = tf.random.normal([len(idx), Z_DIM], seed=seed + 7)
    fake_tr = G([C_tr[idx], oh2, nz2], training=False).numpy()
    rec_train = float((clf.predict(fake_tr, verbose=0).argmax(1) == y_tr[idx]).mean())

    # fidelity: aligned SSIM (C output vs the REAL eval target it was conditioned on)
    ssims = [float(ssim_fn((X_ev[i, :, :, 0] + 1) / 2, (fake_ev[i, :, :, 0] + 1) / 2, data_range=1.0))
             for i in range(len(X_ev))]
    return {"heldout_recognition": rec_heldout, "train_recognition": rec_train,
            "generalization_gap": rec_train - rec_heldout, "fidelity_ssim": float(np.mean(ssims)),
            "_fake_ev": fake_ev, "_C_ev": C_ev}

heldout = heldout_structure_test(generator)
print(f"Recognition on HELD-OUT structures : {heldout['heldout_recognition']:.4f}")
print(f"Recognition on TRAINING structures : {heldout['train_recognition']:.4f}")
print(f"Generalization gap (smaller=better): {heldout['generalization_gap']:.4f}")
print(f"Fidelity SSIM (output vs real)     : {heldout['fidelity_ssim']:.4f}")

In [ ]:
# ── Qualitative held-out grid: unseen structure -> C output -> real target ────
n_show = min(6, len(X_ev))
fig, axes = plt.subplots(3, n_show, figsize=(2.4 * n_show, 7.2))
for j in range(n_show):
    struct = (heldout["_C_ev"][j] * 127.5 + 127.5).clip(0, 255).astype(np.uint8)
    out    = (heldout["_fake_ev"][j, :, :, 0] * 127.5 + 127.5).clip(0, 255).astype(np.uint8)
    real   = (X_ev[j, :, :, 0] * 127.5 + 127.5).clip(0, 255).astype(np.uint8)
    axes[0, j].imshow(struct); axes[0, j].set_title(idx_to_label[y_ev[j]]); axes[0, j].axis('off')
    axes[1, j].imshow(out,  cmap='gray'); axes[1, j].axis('off')
    axes[2, j].imshow(real, cmap='gray'); axes[2, j].axis('off')
for r, lab in enumerate(["unseen structure", "C output", "real target"]):
    axes[r, 0].set_ylabel(lab, fontsize=12, rotation=90, labelpad=12)
    axes[r, 0].axis('on'); axes[r, 0].set_xticks([]); axes[r, 0].set_yticks([])
plt.suptitle("Model C — held-out structure test (generalizes, does not memorize)", y=1.01)
plt.tight_layout(); plt.savefig(os.path.join(PLOTS_DIR, "heldout_C.png"), dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# ── FID (optional — needs pytorch-fid + torch) ────────────────────────────────
fid_m = None
try:
    from pytorch_fid import fid_score
    import torch
    FID_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    def _save_imgs(arr, d):
        os.makedirs(d, exist_ok=True)
        for i, im in enumerate(arr):
            u = (im[:, :, 0] * 127.5 + 127.5).clip(0, 255).astype(np.uint8)
            Image.fromarray(np.stack([u] * 3, -1)).save(os.path.join(d, f"{i:05d}.png"))

    _save_imgs(X_ev, REAL_FID_DIR)
    f_imgs, _ = gen_from_train_structures(generator, n_per=max(N_FID_PER_CLASS, len(X_ev) // num_classes))
    fake_dir = os.path.join(EVAL_DIR, "fid_fake_C"); _save_imgs(f_imgs, fake_dir)
    fid_m = float(fid_score.calculate_fid_given_paths([REAL_FID_DIR, fake_dir], 32, FID_DEVICE, 2048))
    print(f"FID (lower better): {fid_m:.4f}")
except Exception as e:
    print("FID skipped:", e)

## Section 11 — Save Results

In [ ]:
def _nn(obj):
    if isinstance(obj, float) and obj != obj: return None
    if isinstance(obj, dict):  return {k: _nn(v) for k, v in obj.items()}
    if isinstance(obj, list):  return [_nn(i) for i in obj]
    return obj

results_C = {
    "model"      : "cGAN-C 128×128 (Structure-conditioned)",
    "conditioning": "per-image structure map (Canny edge + silhouette + distance transform)",
    "recognition": {"gan_test": rec_m, "classifier_ref_on_real": clf_ref},
    "diversity"  : {"mean": div_m},
    "fid"        : {"mean": fid_m},
    "heldout_structure_test": {
        "heldout_recognition": heldout["heldout_recognition"],
        "train_recognition"  : heldout["train_recognition"],
        "generalization_gap" : heldout["generalization_gap"],
        "fidelity_ssim"      : heldout["fidelity_ssim"],
    },
    "training"   : {"final_d": hist["d"][-1] if hist["d"] else None,
                    "final_g": hist["g"][-1] if hist["g"] else None,
                    "final_g_adv": hist["g_adv"][-1] if hist["g_adv"] else None,
                    "final_g_l1": hist["g_l1"][-1] if hist["g_l1"] else None},
    "hyperparams": {"IMG_SIZE": IMG_SIZE, "Z_DIM": Z_DIM, "COND_CH": COND_CH,
                    "LR_G": LR_G, "LR_D": LR_D, "EPOCHS": EPOCHS, "LAMBDA_L1": LAMBDA_L1},
}

rp = os.path.join(HISTORY_DIR, "C_results.json")
with open(rp, "w") as f: json.dump(_nn(results_C), f, indent=2)
print(f"Results saved: {rp}\n")
print(f"GAN-test recognition : {rec_m:.4f}  (real ref {clf_ref:.4f})")
print(f"Diversity            : {div_m:.4f}")
print(f"FID                  : {fid_m}")
print(f"Held-out recognition : {heldout['heldout_recognition']:.4f}  "
      f"(gap {heldout['generalization_gap']:.4f}, SSIM {heldout['fidelity_ssim']:.4f})")
print()
print("Model C conditions on a per-image structure map (no MediaPipe). The small")
print("held-out generalization gap is the key signal: it generates recognizable")
print("hands from structures it never saw -> it learned structure->image, not copies.")